# Introduction to Retrieval Augmented Generation with S&P 500 news

In this notebook, you will explore how to build a simple Retrieval-Augmented Generation (RAG) pipeline using financial news articles from S&P 500 companies.

We'll start by vectorizing text data, creating a vector store using FAISS, and integrating it with OpenAI's GPT models to answer questions using retrieved information.

This workflow emulates real-world systems in finance where natural language data (news, filings, analyst reports) are used to support decision-making.

# 📌 Objectives

By the end of this notebook, students will be able to:

1. **Perform Semantic Search with Metadata Filtering:**
   - Query the provided FAISS vector store to retrieve relevant financial news articles based on natural language questions.
   - Apply optional filters using metadata such as ticker or publication date to refine search results.

2. **Enrich Data with Company Metadata:**
   - Use the `yfinance` library to retrieve company-level metadata (company name, sector, industry) for tickers in the dataset.
   - Integrate this metadata to support enhanced filtering and analysis of news data.

3. **Build a Retrieval-Augmented Generation (RAG) Pipeline:**
   - Combine retrieved news snippets as context to generate answers using OpenAI’s GPT models.
   - Construct effective prompts that guide the language model to provide concise, context-aware responses.

4. **Evaluate and Analyze RAG Outputs:**
   - Review generated answers alongside the supporting news excerpts.
   - Reflect on the strengths and limitations of the simple RAG pipeline and consider potential improvements, such as adding more filters or refining retrieval strategies.

5. **Incorporate Financial Metadata into Retrieval Context:**
   - Enrich retrieved news snippets with key financial metadata including ticker, company name, sector, and industry.
   - Format prompts that combine both text excerpts and metadata to provide richer context to the language model.

6. **Generate Context-Aware Answers Using OpenAI Models:**
   - Construct and send prompts to an LLM that leverage both news content and metadata to produce concise, informed financial analysis.

7. **Compare Answers With and Without Metadata:**
   - Evaluate the impact of including financial metadata on answer quality using criteria such as clarity, detail, accuracy, and contextual relevance.
   - Summarize findings to reflect on the role of metadata in improving retrieval-augmented generation.

## Install and Import important librairies

First, we install and import the necessary libraries for:
- Text embedding generation (sentence-transformers)
- Efficient similarity search (faiss)
- Data manipulation (pandas, numpy)
- Visualization (matplotlib)

> ℹ️ FAISS uses inner product for cosine similarity by normalizing vectors.

In [ ]:
# Installing needed libraries
%pip install -q sentence-transformers faiss-cpu openai curl_cffi
!pip install yfinance --upgrade --no-cache-dir -q

In [ ]:
# --- Core data handling  ---
import pandas as pd                      # Load and manipulate the news DataFrame
import numpy as np                        # Vector math, normalization of embeddings

# --- Embedding + similarity search ---
from sentence_transformers import SentenceTransformer   # Pre-trained model to turn news text into embeddings
import faiss                                             # Efficient vector index for semantic similarity search
import matplotlib.pyplot as plt                          # For any inline plots/visualizations of results

# --- Financial metadata enrichment ---
import yfinance as yf                     # Fetch company name, sector, industry per ticker
from curl_cffi import requests as curl_requests   # Browser-impersonating session to avoid Yahoo 404/rate-limit blocks on yfinance calls

# --- LLM generation ---
from openai import OpenAI                 # Client to call GPT-4o-mini for RAG answer generation
from google.colab import userdata         # Securely retrieve OPENAI_API_KEY from Colab Secrets

# --- Display / formatting helpers ---
from IPython.display import Markdown, display   # Render generated answers as formatted markdown
import textwrap                                   # Wrap long text (news snippets, answers) for readable printing

# --- Utility ---
import time                               # Add delays between yfinance calls to avoid rate limits

# --- Drive access to csv ---
from google.colab import drive
drive.mount('/content/drive')
csv_path = '/content/drive/MyDrive/Colab Notebooks/Fintech _2/df_news.csv'

# --- GPU access if needed later ---
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
Tesla T4


In [ ]:
# Gettin OpenAI key
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

## Load news data
We load a CSV file of financial news, focusing on TITLE and SUMMARY, along with metadata like TICKER and PUBLICATION_DATE.
These will be embedded into vectors and used for semantic retrieval.

In [ ]:
K = 25

In [ ]:
df_news = pd.read_csv(csv_path)
df_news['PUBLICATION_DATE'] = pd.to_datetime(df_news['PUBLICATION_DATE']).dt.date
display(df_news)

,TICKER,TITLE,SUMMARY,PUBLICATION_DATE,PROVIDER,URL
0,MMM,2 Dow Jones Stocks with Promising Prospects an...,The Dow Jones (^DJI) is made up of 30 of the m...,2025-05-29,StockStory,https://finance.yahoo.com/news/2-dow-jones-sto...
1,MMM,3 S&P 500 Stocks Skating on Thin Ice,The S&P 500 (^GSPC) is often seen as a benchma...,2025-05-27,StockStory,https://finance.yahoo.com/news/3-p-500-stocks-...
2,MMM,3M Rises 15.8% YTD: Should You Buy the Stock N...,"MMM is making strides in the aerospace, indust...",2025-05-22,Zacks,https://finance.yahoo.com/news/3m-rises-15-8-y...
3,MMM,Q1 Earnings Roundup: 3M (NYSE:MMM) And The Res...,Quarterly earnings results are a good time to ...,2025-05-22,StockStory,https://finance.yahoo.com/news/q1-earnings-rou...
4,MMM,3 Cash-Producing Stocks with Questionable Fund...,While strong cash flow is a key indicator of s...,2025-05-19,StockStory,https://finance.yahoo.com/news/3-cash-producin...
...,...,...,...,...,...,...
4866,ZTS,2 Dividend Stocks to Buy With $500 and Hold Fo...,Zoetis is a leading animal health company with...,2025-05-23,Motley Fool,https://www.fool.com/investing/2025/05/23/2-di...
4867,ZTS,Zoetis (NYSE:ZTS) Declares US$0.50 Dividend Pe...,Zoetis (NYSE:ZTS) recently affirmed a dividend...,2025-05-22,Simply Wall St.,https://finance.yahoo.com/news/zoetis-nyse-zts...
4868,ZTS,Jim Cramer on Zoetis (ZTS): “It Does Seem to B...,We recently published a list of Jim Cramer Tal...,2025-05-21,Insider Monkey,https://finance.yahoo.com/news/jim-cramer-zoet...
4869,ZTS,Zoetis (ZTS) Upgraded to Buy: Here's Why,Zoetis (ZTS) might move higher on growing opti...,2025-05-21,Zacks,https://finance.yahoo.com/news/zoetis-zts-upgr...


In [ ]:
df_news['EMBEDDED_TEXT'] = df_news['TITLE'] + ' : ' + df_news['SUMMARY']

In [ ]:
# Check for duplicate articles (same text tagged to multiple tickers)
dupe_mask = df_news.duplicated(subset='EMBEDDED_TEXT', keep=False)
n_dupe_rows = dupe_mask.sum()
n_dupe_articles = df_news.loc[dupe_mask, 'EMBEDDED_TEXT'].nunique()

dupe_groups = df_news[dupe_mask]
date_spread = dupe_groups.groupby('EMBEDDED_TEXT')['PUBLICATION_DATE'].nunique()

print(f"{n_dupe_rows} rows out of {len(df_news)} total involve duplicate text, collapsing to {n_dupe_articles} unique articles")
print(f"{(date_spread == 1).sum()} groups share the same date (same article, multiple tickers); {(date_spread > 1).sum()} span multiple dates")

1023 rows out of 4871 total involve duplicate text, collapsing to 444 unique articles
428 groups share the same date (same article, multiple tickers); 16 span multiple dates


In [ ]:
# Deduplicate — merge same-article, same-date rows tagged to multiple tickers
df_news_dedup = (
    df_news.groupby(['EMBEDDED_TEXT', 'PUBLICATION_DATE'], as_index=False, sort=False)
    .agg({
        'TICKER': lambda x: list(x.unique()),
        'PROVIDER': 'first'
    })
)
print(f"Before: {len(df_news)} rows → After: {len(df_news_dedup)} rows")

Before: 4871 rows → After: 4313 rows


## Implement FAISS vector store
We:
- Use a pre-trained sentence transformer (all-MiniLM-L6-v2) to embed documents.
- Normalize vectors to use cosine similarity.
- Create a FAISS index and implement a basic search function.

This will allow us to retrieve relevant news snippets given a natural language question.


In [ ]:
# Creating both raw and deduplicated versions of documents/metadata,
# to allow direct comparison of retrieval quality before and after deduplication if needed later

# Converting EMBEDDED_TEXT into a list of strings
documents_og = df_news['EMBEDDED_TEXT'].tolist()          # Raw: 4,871 rows, may repeat same article across tickers
documents = df_news_dedup['EMBEDDED_TEXT'].tolist()       # Deduplicated: 4,313 rows, one row per unique (article, date)

# Building the Metadata list
metadata_og = df_news[['PUBLICATION_DATE', 'TICKER', 'PROVIDER']].to_dict(orient='records')       # TICKER is a single string
metadata = df_news_dedup[['PUBLICATION_DATE', 'TICKER', 'PROVIDER']].to_dict(orient='records')    # TICKER is now a list

In [ ]:
metadata_og[0]

{'PUBLICATION_DATE': datetime.date(2025, 5, 29),
 'TICKER': 'MMM',
 'PROVIDER': 'StockStory'}

In [ ]:
# Deduplicated data merges the same article labeled with different tickers into a single row with a list of tickers
metadata[0]

{'PUBLICATION_DATE': datetime.date(2025, 5, 29),
 'TICKER': ['MMM', 'AMGN'],
 'PROVIDER': 'StockStory'}

In [ ]:
# ------ Encode and Normalize  -------

# Load model and compute embeddings
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
embeddings = model.encode(documents, show_progress_bar=True, convert_to_numpy=True)

# Normalize each embedding vector to unit length (L2 norm = 1), to use cosine similarity (via inner product in FAISS)
faiss.normalize_L2(embeddings)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/135 [00:00<?, ?it/s]

In [ ]:
# ----- Initialize the FAISS Index and Add Embeddings -----

dimension = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatIP(dimension) # Cosine similarity via inner product

index.add(embeddings)

In [ ]:
# ---- Reusing my FAISS vector store class from part 2 ----

class FaissVectorStore:
    def __init__(self, model, documents, metadata, index):
        assert len(documents) == len(metadata)
        self.model = model
        self.documents = documents
        self.metadata = metadata
        self.index = index

    def search(self, query, k=5, metadata_filter=None):
        query_embedding = self.model.encode([query]).astype('float32')
        faiss.normalize_L2(query_embedding)

        search_k = k + 10 if metadata_filter is not None else k
        search_k = min(search_k, self.index.ntotal)

        scores, indices = self.index.search(query_embedding, search_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            meta = self.metadata[idx]
            if metadata_filter is not None and not metadata_filter(meta):
                continue
            results.append((self.documents[idx], meta, score))
            if len(results) == k:
                break
        return results

In [ ]:
# create the FAISS-based vector store
faiss_store = FaissVectorStore(model, documents, metadata, index)

### Setup OpenAI Client

👉 **Instructions**:
- Import the `OpenAI` client from the `openai` Python library.
- You will need an **OpenAI API key** to use their models programmatically:
  - Go to [https://platform.openai.com/](https://platform.openai.com/) and sign up or log in.
  - Create an API key from your [API keys dashboard](https://platform.openai.com/account/api-keys).
  - ⚠️ **Keep your API key private** and **do not** share or hardcode it in public notebooks.
- Note that **usage of the OpenAI API is not free**. You will need to:
  - Add a payment method.
  - Monitor your usage to avoid unexpected charges.
  - Optionally set usage limits from your account settings.
- You can refer to the **course’s Study Resources** for a step-by-step guide on creating an OpenAI account and retrieving your API key.

Then:
- Initialize the client with `OpenAI(api_key="YOUR_KEY_HERE")`.
- Send a test request using `.responses.create()` and the `"gpt-4o-mini"` model with a simple prompt:

  ```python
  response = client.responses.create(
      model="gpt-4o-mini",
      input="Write a one-sentence bedtime story about a unicorn."
  )
  print(response.output_text)


In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Write a one-sentence bedtime story about a unicorn."
)
print(response.output_text)

As the stars twinkled like diamonds in the night sky, a gentle unicorn named Luna spread her shimmering wings and soared above the clouds, weaving dreams of magic and wonder for all the children asleep below.


## Retrieve Additional Metadata from Yahoo Finance

👉 **Instructions**:
- We will enrich our news dataset by retrieving **company-level metadata** using the `yfinance` library.
- The goal is to map each unique stock ticker (`TICKER`) in the dataset to:
  - `COMPANY_NAME`
  - `SECTOR`
  - `INDUSTRY`

> ℹ️ `yfinance` fetches live data from Yahoo Finance. If you're running this in a cloud environment or during peak hours, expect some tickers to fail or rate limits to apply.

✅ After this step, you will have a new DataFrame (e.g. `df_meta`) with the columns `TICKER`, `COMPANY_NAME`, `SECTOR`, `INDUSTRY` that maps tickers to their company names, sectors, and industries. This metadata will be useful later to add filters and analysis based on sector or industry categories.


In [ ]:
# Getting unique tickers
unique_tickers = df_news['TICKER'].unique().tolist()
print(f"Number of unique tickers: {len(unique_tickers)}")

Number of unique tickers: 490


In [ ]:
# testing YF API
ticker_obj = yf.Ticker("AAPL")
info = ticker_obj.info
print(info.get('longName'))     # Company name
print(info.get('sector'))       # e.g. "Technology"
print(info.get('industry'))     # e.g. "Consumer Electronics"

Apple Inc.
Technology
Consumer Electronics


In [ ]:
# Fetch company name, sector, and industry for every unique ticker in the dataset.

# Use a browser-impersonating session (curl_cffi) to reduce Yahoo Finance 404/rate-limit errors
session = curl_requests.Session(impersonate="chrome")

# Each field falls back to 'Unknown' independently, so a partial response (e.g. missing
# only the company name) doesn't leave a raw None sitting in the DataFrame.

records = []
failed_tickers = []

for ticker in unique_tickers:
    try:
        # Fetch metadata for this ticker using the Chrome-impersonating session
        info = yf.Ticker(ticker, session=session).info

        # Fall back to 'Unknown' per field, independently, instead of an all-or-nothing check
        name = info.get('longName') or 'Unknown'
        sector = info.get('sector') or 'Unknown'
        industry = info.get('industry') or 'Unknown'

        # Only count it as a full failure if every field came back empty
        if name == 'Unknown' and sector == 'Unknown' and industry == 'Unknown':
            failed_tickers.append(ticker)

        records.append({
            'TICKER': ticker,
            'COMPANY_NAME': name,
            'SECTOR': sector,
            'INDUSTRY': industry
        })

    except Exception as e:
        # Catches network errors, HTTP errors, or anything yfinance itself raises
        print(f"Failed for {ticker}: {e}")
        failed_tickers.append(ticker)
        records.append({
            'TICKER': ticker,
            'COMPANY_NAME': 'Unknown',
            'SECTOR': 'Unknown',
            'INDUSTRY': 'Unknown'
        })

    # Small delay between requests to reduce the chance of rate limiting
    time.sleep(0.2)

# Build the final metadata DataFrame, one row per unique ticker
df_meta = pd.DataFrame(records)

# Report how many tickers had no usable data at all, for the written analysis
print(f"Fully failed to retrieve metadata for {len(failed_tickers)} tickers: {failed_tickers}")

# Sanity check: confirm no raw None values remain in the key columns
print(df_meta[['COMPANY_NAME', 'SECTOR', 'INDUSTRY']].isna().sum())

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BK"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CTRA"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DAY"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: IPG"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: K"}}}
ERROR:yfinance:

Fully failed to retrieve metadata for 11 tickers: ['ANSS', 'BK', 'CTRA', 'DAY', 'FI', 'HOLX', 'IPG', 'JNPR', 'K', 'MMC', 'WBA']
COMPANY_NAME    0
SECTOR          0
INDUSTRY        0
dtype: int64


In [ ]:
# Cheking failed tickers in DF
df_meta[df_meta['TICKER'].isin(['ANSS', 'BK', 'CTRA'])]

,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
34,ANSS,Unknown,Unknown,Unknown
64,BK,Unknown,Unknown,Unknown
125,CTRA,Unknown,Unknown,Unknown


In [ ]:

df_meta.head(10)

,TICKER,COMPANY_NAME,SECTOR,INDUSTRY
0,MMM,3M Company,Industrials,Conglomerates
1,AOS,A. O. Smith Corporation,Industrials,Specialty Industrial Machinery
2,ABT,Abbott Laboratories,Healthcare,Medical Devices
3,ABBV,AbbVie Inc.,Healthcare,Drug Manufacturers - General
4,ACN,Accenture plc,Technology,Information Technology Services
5,ADBE,Adobe Inc.,Technology,Software - Application
6,AMD,"Advanced Micro Devices, Inc.",Technology,Semiconductors
7,AES,The AES Corporation,Utilities,Utilities - Diversified
8,AFL,Aflac Incorporated,Financial Services,Insurance - Life
9,A,"Agilent Technologies, Inc.",Healthcare,Diagnostics & Research


In [ ]:
df_meta.shape

(490, 4)

## Retrieval-Augmented Generation (RAG): Retrieve Documents and Generate Answers

👉 **Instructions**:

In this part of the assignment, your task is to build a simple Retrieval-Augmented Generation (RAG) pipeline that:

- Takes a user question as input.
- Searches the FAISS vector store to find a set of relevant financial news articles based on semantic similarity.
- Uses the retrieved news articles as context to generate a clear, concise answer to the question by interacting with the OpenAI language model.
- Returns both the generated answer and the underlying news snippets used for context.

### What you need to focus on:

- Implement a retrieval mechanism to query your vector store and obtain the top relevant documents for any question.
- Construct prompts that effectively combine retrieved news content with the user’s question to guide the language model’s response.
- Use the OpenAI API to generate answers grounded in the retrieved context.
- Organize the outputs so that for each question, you have:
  - The generated answer.
  - The collection of news excerpts used to produce that answer.

### What you will be provided:

- Helper functions to display outputs in markdown format.
- Lists of example questions covering topics, companies, and industries to test your implementation.

---

Your solution can take any form or structure you find appropriate, as long as it fulfills these core objectives. This exercise will give you hands-on experience with integrating retrieval and generation for practical applications in finance.


#### Print markdown
You can use the following function to print answers from GPT4o-mini in markdown.

In [ ]:
def print_markdown(text):
    display(Markdown(text))

#### Predefined questions

In [ ]:
questions_topic = [
"What are the major concerns expressed in financial news about inflation?",
"How is investor sentiment described in recent financial headlines?",
"What role is artificial intelligence playing in recent finance-related news stories?"
]

questions_company = [
"How is Microsoft being portrayed in news stories about artificial intelligence?",
"What financial news headlines connect Amazon with automation or logistics?"
]

questions_industry = [
"What are the main themes emerging in financial news about the semiconductor industry?",
"What trends are being reported in the retail industry?",
"What risks or challenges are discussed in recent news about the energy industry?"
]

In [ ]:
def generate_answer (question, k=5, metadata_filter=None):

  # Retrieve relevant news snippets from FAISS store
  results = faiss_store.search(question, k, metadata_filter)

  # Building context text from retrieved documents
  context = "\n\n".join([f"- {doc}" for doc, meta, score in results])

  # Constructing the prompt: context + question
  prompt = f"""Answer the following question using ONLY the news excerpts below as context.

                Context:
                {context}

                Question: {question}

                Answer:"""

  # Send prompt to GPT-4o-mini
  response = client.responses.create(
      model = "gpt-4o-mini",
      input = prompt
  )

  return response.output_text, results, context


In [ ]:
# Testing function on one example

answer, results, context = generate_answer(questions_topic[0])

print_markdown(f"**Question:** {questions_topic[0]}")
print_markdown(f"**Answer:**{answer}")
print_markdown(f"**Context:**\n\n{context}")


print_markdown("**Retrieved snippets (score | ticker | date):**")
for doc, meta, score in results:
    ticker = meta['TICKER'] if isinstance(meta['TICKER'], str) else ', '.join(meta['TICKER'])
    print_markdown(f"- {score:.3f} | {ticker} | {meta['PUBLICATION_DATE']}")

**Question:** What are the major concerns expressed in financial news about inflation?

**Answer:**The major concerns expressed in financial news about inflation include persistent US inflation risks noted by the Federal Reserve, which signal potential economic slowdown. Additionally, food inflation is causing increased grocery bills, dampening hopes for interest rate cuts due to ongoing tariff issues.

**Context:**

- Bitcoin price slips as Fed minutes flag US inflation risks : The Federal Reserve’s May policy meeting revealed mounting concern over persistent US inflation and the potential for economic slowdown.

- The Weekend: Food inflation dampens hopes of a rate cut as tariff twists and turns continue : Key moments from the last seven days, plus a glimpse at the week ahead

- 8 Best Grocery Items at Dollar Tree To Help Combat Inflation This Summer : Once again, creeping inflation and new tariffs have caused your weekly grocery bills to seemingly skyrocket. Popular items like orange juice, eggs, chicken breasts, fresh ground beef, bacon, seafood,...

- We Like These Underlying Return On Capital Trends At Boston Scientific (NYSE:BSX) : What trends should we look for it we want to identify stocks that can multiply in value over the long term? One common...

- 3 of Wall Street’s Favorite Stocks Facing Headwinds : Wall Street has set ambitious price targets for the stocks in this article. While this suggests attractive upside potential, it’s important to remain skeptical because analysts face institutional pressures that can sometimes lead to overly optimistic forecasts.

**Retrieved snippets (score | ticker | date):**

- 0.577 | BLK | 2025-05-29

- 0.492 | AVGO, LULU, NVDA, TSLA | 2025-05-31

- 0.477 | DLTR | 2025-05-29

- 0.453 | BSX | 2025-05-30

- 0.453 | KMX | 2025-05-26

In [ ]:
all_questions = questions_topic + questions_company + questions_industry

results_log = []

for question in all_questions:
    answer, results, context = generate_answer(question)
    results_log.append({
        'question': question,
        'answer': answer,
        'context': context,
        'results': results
    })

    print_markdown(f"**Question:** {question}")
    print_markdown(f"**Answer:** {answer}")
    print_markdown(f"**Context:**\n\n{context}")

    print_markdown("**Retrieved snippets (score | ticker | date):**")
    for doc, meta, score in results:
        ticker = meta['TICKER'] if isinstance(meta['TICKER'], str) else ', '.join(meta['TICKER'])
        print_markdown(f"- {score:.3f} | {ticker} | {meta['PUBLICATION_DATE']}")

    print_markdown("---")

**Question:** What are the major concerns expressed in financial news about inflation?

**Answer:** The major concerns expressed in financial news about inflation include persistent US inflation risks, as noted by the Federal Reserve's minutes highlighting potential economic slowdown. Additionally, food inflation is dampening hopes for a rate cut, and new tariffs are causing grocery bills to increase significantly, indicating ongoing inflationary pressures.

**Context:**

- Bitcoin price slips as Fed minutes flag US inflation risks : The Federal Reserve’s May policy meeting revealed mounting concern over persistent US inflation and the potential for economic slowdown.

- The Weekend: Food inflation dampens hopes of a rate cut as tariff twists and turns continue : Key moments from the last seven days, plus a glimpse at the week ahead

- 8 Best Grocery Items at Dollar Tree To Help Combat Inflation This Summer : Once again, creeping inflation and new tariffs have caused your weekly grocery bills to seemingly skyrocket. Popular items like orange juice, eggs, chicken breasts, fresh ground beef, bacon, seafood,...

- We Like These Underlying Return On Capital Trends At Boston Scientific (NYSE:BSX) : What trends should we look for it we want to identify stocks that can multiply in value over the long term? One common...

- 3 of Wall Street’s Favorite Stocks Facing Headwinds : Wall Street has set ambitious price targets for the stocks in this article. While this suggests attractive upside potential, it’s important to remain skeptical because analysts face institutional pressures that can sometimes lead to overly optimistic forecasts.

**Retrieved snippets (score | ticker | date):**

- 0.577 | BLK | 2025-05-29

- 0.492 | AVGO, LULU, NVDA, TSLA | 2025-05-31

... (70 outputs left out)

- 0.478 | FCX | 2025-05-21

- 0.472 | SLB | 2025-05-10

- 0.472 | D | 2025-05-22

---

## Analysis & Questions - Section 1

### Analysis and Reflection on Retrieval and Generation Results
After running the RAG pipeline and obtaining answers along with their supporting news excerpts, take some time to carefully review both the generated responses and the retrieved contexts.

- **For each question, read the answer and then the corresponding news snippets used as context.**

- Reflect on the following points and document your observations:
1. **Relevance**
2. **Completeness**  
3. **Bias or Noise**
4. **Consistency**  
5. **Improvement Ideas**   

and answer the questions below:

#### **Question 1.** How well do the retrieved news snippets support the generated answer? Are the key facts or themes in the answer clearly grounded in the context?

> Overall, grounding was strong for topic- and industry-focused questions. The inflation answer correctly cited three distinct facts traceable to specific snippets: persistent US inflation flagged by Fed minutes (score 0.577), food inflation dampening rate-cut hopes, and tariffs raising grocery bills (Dollar Tree snippet, 0.477). The AI, semiconductor, and energy answers showed similarly tight grounding, with every claim mapping cleanly to a retrieved snippet (e.g., the energy answer's four distinct risk categories each traced to a separate article, scores 0.47–0.54).

> The investor-sentiment answer is a notable exception: while it sounds coherent, all five retrieved snippets (scores 0.57–0.61) were the same recurring boilerplate template ("X of Wall Street's Favorite Stocks...") applied to unrelated companies (KMX, MCHP, MPWR, DRI, RVTY). The answer is technically grounded in the text, but the text itself contains no real sentiment analysis of these specific companies, it's a generic analyst-disclaimer format, not genuine sentiment reporting.


#### **Question 2.** Does the answer fully address the question, or does it leave important aspects out? Consider if the retrieved context provided enough information to generate a thorough response.

  > Most answers were reasonably complete given the available context. The clearest completeness failure was the Microsoft question: retrieval returned zero snippets mentioning Microsoft (scores 0.46–0.54, topped by META and CRM), and the model correctly declined to answer rather than hallucinate: "The excerpts provided do not mention Microsoft or describe its portrayal in relation to artificial intelligence." This is a retrieval failure, not a generation failure, the underlying dataset likely contains Microsoft/AI articles, but this specific query and k=5 didn't surface any of them.

> The Amazon/logistics answer also showed a partial completeness gap: across repeated runs, it consistently had to stretch to include a C.H. Robinson logistics article that doesn't actually mention Amazon, explicitly flagging "while not directly mentioning Amazon", a sign the retrieved context didn't fully cover the question's second half (logistics), even though it covered automation (AWS/Garman AI roadmap snippet, 0.523).

#### **Question 3.** Are there any irrelevant or misleading snippets retrieved that may have influenced the answer? How might this affect the quality of the output?

> Yes, and this was the most consistent pattern across the batch. A specific templated article genre, generic "stock trend/return on capital" boilerplate with company names swapped in, repeatedly surfaced across unrelated questions: Boston Scientific and CarMax in the inflation results, and Packaging Corp/Air Products in the retail results (all scoring 0.45–0.48).


> More concerning, the entire investor-sentiment result set consisted entirely of this same templated genre, scoring deceptively high (0.57–0.61) despite containing no company-specific sentiment content. This shows that boilerplate financial-writing style can score as "relevant" as genuine analysis, since it's the language pattern, not the substance, driving the embedding similarity. This is a systemic quirk of the underlying news dataset, likely tied to StockStory's recurring column formats, rather than a one-off retrieval error.


#### **Question 4.**  Do the news snippets show consistent information, or are there conflicting viewpoints? How does the LLM handle potential contradictions in the context?

> Retrieval proved to be completely deterministic: re-running all 8 questions three separate times returned identical similarity scores and tickers every single time (e.g., inflation's top result stayed at 0.577/BLK across all three runs). This confirms the FAISS search step introduces zero variability on its own, meaning any inconsistency in the pipeline's output has to come from the generation step, not retrieval.

> Generation, by contrast, showed real variability despite receiving identical context each time. Most answers only varied in phrasing or structure (paragraph vs. bulleted list), but the Amazon/logistics question revealed a more meaningful inconsistency: it produced a conservative 2-point answer in two of the three runs, but in one run added a third, more speculative point that stretched an unrelated Woodward's automation snippet into supporting evidence for Amazon. No direct factual contradictions appeared across any snippets in the retrieved context itself; the inconsistency was specifically in how confidently the LLM was willing to bridge gaps in weak context, not in the source material disagreeing with itself.  

#### **Question 5.**  Based on your observations, suggest ways the retrieval or generation process could be improved (e.g., better filtering, adjusting `k`, refining prompt design).

> Two concrete, evidence-based improvements emerged from testing:

> - **Similarity-score threshold**: for the inflation question, on-topic snippets scored 0.477+, while off-topic noise scored exactly 0.453. A filter around 0.46 would have removed the Boston Scientific/CarMax noise while keeping relevant results.

> - **Template-aware filtering or prompt guidance**: the investor-sentiment case shows a threshold alone isn't enough, since boilerplate content scored higher than some genuinely relevant snippets elsewhere. A stronger fix would be adding an explicit prompt instruction (e.g., "ignore generic disclaimer/template language not specific to named companies") or deduplicating near-identical templated openers across different tickers before retrieval, similar to the exact-duplicate dedup already applied to EMBEDDED_TEXT.

> - **Increasing k with reranking** for low-recall cases like the Microsoft question would also help retrieving a larger candidate pool (e.g., k=15) and letting a secondary relevance check filter down to the best 5 could recover genuinely relevant but lower-ranked results.
  

## 🧠 Retrieval-Augmented Generation (RAG) v2: Adding Financial Metadata to Improve Generation

👉 **Instructions**:

In this part of the assignment, you’ll enhance your Retrieval-Augmented Generation (RAG) pipeline by incorporating *financial metadata* to provide more contextually rich answers.

Your goal is to evaluate whether metadata such as **company name**, **sector**, and **industry** helps the LLM generate **more accurate and grounded answers** to financial questions.

---

### ✅ What your updated pipeline should do:

- Retrieve relevant financial news articles using semantic similarity with FAISS.
- Enrich each retrieved document with financial metadata:
  - Ticker symbol
  - Full company name
  - Sector (e.g., Technology, Energy)
  - Industry (e.g., Semiconductors, Retail)
- Construct prompts that include both:
  - Retrieved news text
  - Associated metadata
- Send the prompt to the OpenAI model to generate an informed response.
- Return:
  - The final answer
  - The exact set of contextual documents used to produce that answer

---

### 🧪 Evaluation and Comparison:

You will test your improved RAG pipeline on the same three types of questions provided earlier:
- **Topic-focused** (e.g., inflation, interest rates)
- **Company-focused** (e.g., questions about Tesla, Nvidia)
- **Industry-focused** (e.g., semiconductors, utilities)


In [ ]:
# Building the Lookup Dictionary
ticker_lookup = df_meta.set_index('TICKER').to_dict(orient='index')

# Testing on Apple ticker
ticker_lookup['AAPL']

{'COMPANY_NAME': 'Apple Inc.',
 'SECTOR': 'Technology',
 'INDUSTRY': 'Consumer Electronics'}

In [ ]:
# Enrich with metadata function

def enrich_snippet(doc, meta):
    # meta['TICKER'] can be a single string (undeduped articles) or a list
    # (articles merged during dedup, tagged to multiple tickers). Normalize
    # both cases into a list so the loop below always works the same way.
    tickers = meta['TICKER'] if isinstance(meta['TICKER'], list) else [meta['TICKER']]

    enrichment = []
    for t in tickers:
        # Look up this ticker's company info from the prebuilt dictionary.
        # If the ticker somehow isn't in ticker_lookup, fall back to 'Unknown'
        # for every field instead of raising a KeyError.
        info = ticker_lookup.get(t, {'COMPANY_NAME': 'Unknown', 'SECTOR': 'Unknown', 'INDUSTRY': 'Unknown'})

        # Format this one ticker's info as "TICKER (Company Name, Sector/Industry)"
        enrichment.append(f"{t} ({info['COMPANY_NAME']}, {info['SECTOR']}/{info['INDUSTRY']})")

    # Join all ticker entries with "; " in case there are multiple,
    # then prefix the article text with this bracketed metadata block
    return f"- [{'; '.join(enrichment)}] {doc}"

In [ ]:
# testing the metadata enrichment

results = faiss_store.search("NVIDIA earnings", k=1)
doc, meta, score = results[0]
print(meta)
print()
print(enrich_snippet(doc, meta))

{'PUBLICATION_DATE': datetime.date(2025, 5, 31), 'TICKER': ['NVDA'], 'PROVIDER': 'Simply Wall St.'}

- [NVDA (NVIDIA Corporation, Technology/Semiconductors)] NVIDIA First Quarter 2026 Earnings: Revenues Beat Expectations, EPS Lags : NVIDIA ( NASDAQ:NVDA ) First Quarter 2026 Results Key Financial Results Revenue: US$44.1b (up 69% from 1Q 2025). Net...


In [ ]:

def generate_answer_v2(question, k=5, metadata_filter=None):

    # Retrieve relevant news snippets from FAISS store
    results = faiss_store.search(question, k, metadata_filter)

    # Building enriched context text with ticker/company/sector/industry metadata
    context = "\n\n".join([enrich_snippet(doc, meta) for doc, meta, score in results])

    # Constructing the prompt: enriched context + question
    prompt = f"""Answer the following question using ONLY the news excerpts and associated
company metadata below as context.

                Context:
                {context}

                Question: {question}

                Answer:"""

    # Send prompt to GPT-4o-mini
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text, results, context

In [ ]:
all_questions = questions_topic + questions_company + questions_industry

results_log_v2 = []

for question in all_questions:
    answer, results, context = generate_answer_v2(question)
    results_log_v2.append({
        'question': question,
        'answer': answer,
        'context': context,
        'results': results
    })

    print_markdown(f"**Question:** {question}")
    print_markdown(f"**Answer:** {answer}")
    print_markdown(f"**Context:**\n\n{context}")

    print_markdown("**Retrieved snippets (score | ticker | company/sector | date):**")
    for doc, meta, score in results:
        ticker = meta['TICKER'] if isinstance(meta['TICKER'], str) else ', '.join(meta['TICKER'])
        tickers_list = meta['TICKER'] if isinstance(meta['TICKER'], list) else [meta['TICKER']]
        info_str = ', '.join([
            f"{ticker_lookup.get(t, {'COMPANY_NAME': 'Unknown', 'SECTOR': 'Unknown'})['COMPANY_NAME']} ({ticker_lookup.get(t, {'SECTOR': 'Unknown'})['SECTOR']})"
            for t in tickers_list
        ])
        print_markdown(f"- {score:.3f} | {ticker} | {info_str} | {meta['PUBLICATION_DATE']}")

    print_markdown("---")

**Question:** What are the major concerns expressed in financial news about inflation?

**Answer:** The major concerns expressed in financial news about inflation include persistent US inflation risks flagged by the Federal Reserve, which could lead to a potential economic slowdown. Additionally, food inflation is dampening hopes for a rate cut, and new tariffs are driving up grocery bills, impacting consumer spending. Overall, there is apprehension over inflation's effects on economic stability and market expectations.

**Context:**

- [BLK (BlackRock, Inc., Financial Services/Asset Management)] Bitcoin price slips as Fed minutes flag US inflation risks : The Federal Reserve’s May policy meeting revealed mounting concern over persistent US inflation and the potential for economic slowdown.

- [AVGO (Broadcom Inc., Technology/Semiconductors); LULU (lululemon athletica inc., Consumer Cyclical/Apparel Retail); NVDA (NVIDIA Corporation, Technology/Semiconductors); TSLA (Tesla, Inc., Consumer Cyclical/Auto Manufacturers)] The Weekend: Food inflation dampens hopes of a rate cut as tariff twists and turns continue : Key moments from the last seven days, plus a glimpse at the week ahead

- [DLTR (Dollar Tree, Inc., Consumer Defensive/Discount Stores)] 8 Best Grocery Items at Dollar Tree To Help Combat Inflation This Summer : Once again, creeping inflation and new tariffs have caused your weekly grocery bills to seemingly skyrocket. Popular items like orange juice, eggs, chicken breasts, fresh ground beef, bacon, seafood,...

- [BSX (Boston Scientific Corporation, Healthcare/Medical Devices)] We Like These Underlying Return On Capital Trends At Boston Scientific (NYSE:BSX) : What trends should we look for it we want to identify stocks that can multiply in value over the long term? One common...

- [KMX (CarMax, Inc., Consumer Cyclical/Auto & Truck Dealerships)] 3 of Wall Street’s Favorite Stocks Facing Headwinds : Wall Street has set ambitious price targets for the stocks in this article. While this suggests attractive upside potential, it’s important to remain skeptical because analysts face institutional pressures that can sometimes lead to overly optimistic forecasts.

**Retrieved snippets (score | ticker | company/sector | date):**

- 0.577 | BLK | BlackRock, Inc. (Financial Services) | 2025-05-29

- 0.492 | AVGO, LULU, NVDA, TSLA | Broadcom Inc. (Technology), lululemon athletica inc. (Consumer Cyclical), NVIDIA Corporation (Technology), Tesla, Inc. (Consumer Cyclical) | 2025-05-31

... (70 outputs left out)

- 0.478 | FCX | Freeport-McMoRan Inc. (Basic Materials) | 2025-05-21

- 0.472 | SLB | SLB N.V. (Energy) | 2025-05-10

- 0.472 | D | Dominion Energy, Inc. (Utilities) | 2025-05-22

---

## Analysis & Questions - Section 2

### Instructions: Evaluate Answers With and Without Metadata

For each question, compare the two answers provided:
- One generated **without** metadata
- One generated **with** metadata

---

### Steps:

1. Use the following evaluation criteria:
   - Clarity
   - Detail & Depth
   - Use of Context
   - Accuracy & Grounding
   - Relevance
   - Narrrative Flow

2. For each criterion, write brief notes comparing how the answer **without metadata** performs versus the answer **with metadata**.

3. Summarize your evaluation in a markdown table with the following columns:

| Criteria       | WITHOUT METADATA            | WITH METADATA             |
|----------------|----------------------------|--------------------------|
| Clarity        | [Your brief note here]     | [Your brief note here]   |
| Detail & Depth         | [Your brief note here]     | [Your brief note here]   |
| Use of Context        | [Your brief note here]     | [Your brief note here]   |
| Accuracy & Grounding       | [Your brief note here]     | [Your brief note here]   |
| Relevance      | [Your brief note here]     | [Your brief note here]   |
| Narrative Flow      | [Your brief note here]     | [Your brief note here]   |

---

**Note:** Keep comments short and clear for easy comparison.





| Criteria | WITHOUT METADATA | WITH METADATA |
|---|---|---|
| Clarity | Reads as flat paragraphs; investor sentiment answer refers vaguely to "some stocks" without naming them. | Same core content but often broken into labeled bullets (e.g., semiconductor/energy answers use headers like "Investor Interest," "Legislative Threats"), making points easier to scan. |
| Detail & Depth | Generally accurate but less specific; e.g., AI answer references "Upstart" even though it wasn't in the top-5 retrieved snippets, a minor grounding slip. | Slightly more specific; investor sentiment answer explicitly names Microchip Technology, Revvity, CarMax, and Darden Restaurants instead of speaking generically. |
| Use of Context | Uses raw article text only; no way to trace which company or sector backs each claim. | Explicitly cites tickers inline after claims (e.g., "(BLK)," "(DLTR)"), making it easy to verify which snippet supports which point. |
| Accuracy & Grounding | Mostly grounded, but doesn't distinguish sector-relevant snippets from noise (e.g., BSX/KMX folded into the inflation answer without flagging irrelevance). | Same underlying grounding; metadata doesn't fix retrieval noise but makes mismatches visible faster (e.g., you can see BSX is Healthcare, unrelated to inflation, right in the context). |
| Relevance | Struggles equally on weak-retrieval questions (Amazon/logistics, Microsoft/AI) since the issue is retrieval, not the prompt. | No real improvement on the same weak-retrieval questions; v2 answer even adds extra hedging language ("indirectly connects...") to stretch a thin Amazon/logistics context. |
| Narrative Flow | Single flowing paragraphs for most answers; less structured for multi-point questions like semiconductors and energy risks. | More structured, multi-bullet format for the same content, giving a clearer sense of distinct themes within one answer. |

**Takeaway:** Metadata enrichment improves structure, traceability, and specificity, but does not compensate for weak retrieval when the underlying snippets aren't actually relevant to the question.
